# 第6回　標本分布と中心極限定理の再考
## ―― CLTの魔法には「独立」という隠れた前提がある

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

統計学Ⅰで「中心極限定理（CLT）」を習った ―― 元の分布がどんな形でも、標本平均は正規分布に近づく、という魔法だ。Ⅱでは、その魔法の **契約書の細かい字** を読む。そこには **「標本は独立であること」** と書いてある。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
print("準備OK。次のセルへ。")

---
## 1. CLTの復習 ―― 歪んだ母集団でも、標本平均は正規になる

まず母集団を、思いきり歪ませる。「待ち時間」のような、左に偏って右に長い裾を引く分布（指数分布）を使おう。これは正規分布とは似ても似つかない。

この母集団から **独立に** $n=30$ 人を選んで平均を取る、を何千回もやると、標本平均の分布はどうなるか？　予想を決めてから ▶。

In [ ]:
rng = np.random.default_rng(6)
母集団 = rng.exponential(scale=10, size=1_000_000)   # 平均10・右に長い裾（正規でない）
母平均, 母SD = 母集団.mean(), 母集団.std()

plt.figure(figsize=(7, 3.5))
plt.hist(母集団, bins=80, density=True, color="#bcd", edgecolor="none")
plt.title(f"母集団（指数分布）：平均{母平均:.1f}, SD{母SD:.1f} ― まったく正規でない")
plt.xlabel("値"); plt.ylabel("密度"); plt.xlim(0, 60)
plt.show()

In [ ]:
# 独立に n=30 を選んで平均、を1万回くりかえす
n, 回数 = 30, 10000
標本平均 = 母集団[rng.integers(0, len(母集団), size=(回数, n))].mean(axis=1)

理論SE = 母SD / np.sqrt(n)        # CLTが予言する標本平均のばらつき
plt.figure(figsize=(7, 4))
plt.hist(標本平均, bins=50, density=True, color="#3949ab", edgecolor="none")
plt.axvline(母平均, color="#e8503a", lw=2, label=f"母平均={母平均:.1f}")
plt.title(f"標本平均(n=30)の分布 ― きれいな正規！\n実測SD={標本平均.std():.3f}　理論SE=σ/√n={理論SE:.3f}")
plt.xlabel("標本平均"); plt.ylabel("密度"); plt.legend()
plt.show()
print(f"母集団は歪んでいたのに、標本平均の分布は正規に近い。")
print(f"そのばらつき（標準誤差）も理論 σ/√n = {理論SE:.3f} とほぼ一致：実測 {標本平均.std():.3f}")

**母集団はあれほど歪んでいたのに、標本平均はきれいな正規分布**になった。これが中心極限定理だ。

そして標本平均のばらつき（**標準誤差 SE**）は理論どおり

$$\mathrm{SE} = \frac{\sigma}{\sqrt{n}}$$

になっている。$n$ を増やせば SE は小さくなる＝推定が精密になる。**ここまではⅠの復習。**

---
## 2. 契約書の細かい字 ―― CLTは「独立」を前提にしている

いま、さらっと **「独立に $n=30$ 人を選んで」** と書いた。実は、CLTも標準誤差の式 $\sigma/\sqrt{n}$ も、**標本が独立**であることを前提にしている。

では、もし標本が独立でなかったら？　例えばアンケートで、回答者が **隣の人の答えを気にして似た回答をする**（空気を読む）。一人ひとりは母集団から来ているが、互いに **相関** している。

このとき「標本平均のばらつき」は、独立を仮定した $\sigma/\sqrt{n}$ と同じだろうか？　予想を決めてから ▶。

In [ ]:
# 相関のある標本を作る：各標本内の n 人が相関 ρ で似通う（空気を読む度合い）
# X_i = sqrt(ρ)·共通成分 + sqrt(1-ρ)·個人成分  → どの2人も相関ρ、各自の分散は1
def 標本平均のばらつき(n, ρ, 回数=20000):
    共通 = rng.normal(size=(回数, 1))
    個人 = rng.normal(size=(回数, n))
    標本 = np.sqrt(ρ) * 共通 + np.sqrt(1 - ρ) * 個人   # 母SD=1
    return 標本.mean(axis=1).std()                       # 標本平均の実際のばらつき

n = 30
素朴なSE = 1 / np.sqrt(n)                 # 独立を仮定した σ/√n （σ=1）
独立の実測 = 標本平均のばらつき(n, ρ=0.0)
相関の実測 = 標本平均のばらつき(n, ρ=0.3)
print(f"n={n}, 母SD=1 のとき")
print(f"  独立を仮定した素朴なSE σ/√n      ： {素朴なSE:.3f}")
print(f"  実際のばらつき（独立な標本 ρ=0）  ： {独立の実測:.3f}  ← 一致。CLT通り")
print(f"  実際のばらつき（相関した標本 ρ=0.3）： {相関の実測:.3f}  ← 約3倍に膨らむ！")
print(f"\n→ 空気を読む(相関する)と、本当の不確実性は素朴なSEの {相関の実測/素朴なSE:.1f} 倍。")
print(f"   素朴なSEは『実際よりずっと精密だ』と嘘をつく。")

**独立なら理論どおり。だが相関すると、標本平均の本当のばらつきは素朴な $\sigma/\sqrt{n}$ の約3倍に膨らむ。**

つまり「空気を読んだ」標本では、推定は実際よりずっと不確実なのに、$\sigma/\sqrt{n}$ を信じると **過剰に自信を持ってしまう**。検定なら偽の有意差、区間推定なら狭すぎる信頼区間（→第7回）につながる。

---
## 3. 最大のワナ ―― 「nを増やせば解決」は通用しない

「相関が問題なら、サンプルを増やせばいいのでは？」と思うかもしれない。独立なら確かに $n$ を増やすほど SE は0に近づく。

だが **相関した標本では、$n$ をいくら増やしても不確実性は0にならない**。これを確かめよう。

In [ ]:
nリスト = [10, 30, 100, 300, 1000, 3000]
独立 = [標本平均のばらつき(n, ρ=0.0) for n in nリスト]
相関 = [標本平均のばらつき(n, ρ=0.3) for n in nリスト]

plt.figure(figsize=(7, 4))
plt.plot(nリスト, 独立, "o-", color="#3949ab", label="独立な標本（ρ=0）")
plt.plot(nリスト, 相関, "s-", color="#e8503a", label="相関した標本（ρ=0.3＝空気を読む）")
plt.axhline(np.sqrt(0.3), ls="--", color="gray", label="√ρ ≈ 0.55（下げ止まり）")
plt.xscale("log")
plt.xlabel("標本サイズ n（対数）"); plt.ylabel("標本平均の本当のばらつき")
plt.title("独立なら n とともに0へ。相関すると下げ止まる")
plt.legend()
plt.show()
print(f"n=3000 でも：独立 {独立[-1]:.3f}（ほぼ0）／相関 {相関[-1]:.3f}（√0.3≈0.55 から下がらない）")

独立な標本（青）は $n$ とともに0へ落ちる ―― データを増やすほど精密になる。

だが相関した標本（赤）は $\sqrt{\rho}\approx0.55$ で **下げ止まる**。3000人集めても、10人のときとほとんど変わらない。

> 💬 **今期の核心**
> 
> みんなが空気を読んで似た答えをすると、**いくら人数を集めても、独立な少人数ぶんの情報しか得られない**。「大勢が賛成しているから確かだ」は、その大勢が互いを見て合わせているなら、まったく根拠にならない。これが第13回 **コンドルセの陪審定理** ―― 「独立な多数は賢いが、空気を読む多数は賢くない」 ―― の数理的な正体だ。


---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| CLT | 母集団が歪んでいても、**独立な**標本平均は正規分布に近づく |
| 標準誤差 | $\mathrm{SE}=\sigma/\sqrt{n}$。ただし**独立が前提** |
| 独立が崩れると | 本当のばらつきが SE より大きい→過剰な自信→偽の有意差・狭すぎる区間 |
| nを増やしても | 相関は消えない。不確実性は $\sqrt{\rho}$ で下げ止まる |

- CLTは魔法だが、契約書には「標本は独立」と書いてある。
- 「空気を読む」標本は、人数のわりに情報が乏しい。**大人数＝信頼できる、ではない**。

> **課題（Moodle）**：CLTの前提の説明（自動採点）＋「標本が独立でないと、推測は具体的に何を間違えるか」の記述。詳しくはMoodleの第6回課題を見ること。

> **次回予告**：第7回「区間推定の理論」。今日見た標準誤差から、信頼区間がどう作られるか。そして **独立でない標本では、信頼区間が狭くなりすぎて『95%』が嘘になる** ことを確かめる。